# Day 55 — APIs & Containerization: Dockerize a prediction service
Objectives:
- Package your FastAPI model API (from Day 44).
- Create a minimal Dockerfile.
- Build and run locally; test with curl.
Note: You need Docker Desktop or a Docker runtime installed to build images.

## Project layout
````
ds-60day/
  app.py               # from Day 44 (FastAPI)
  model.joblib         # saved model
  requirements-api.txt # smaller requirements for API runtime
  Dockerfile
````

## Example requirements-api.txt
```text
fastapi
uvicorn
joblib
numpy
scikit-learn
pydantic
```


## Example Dockerfile
```Dockerfile
# syntax=docker/dockerfile:1
FROM python:3.12-slim
WORKDIR /app
COPY requirements-api.txt ./
RUN python -m pip install --no-cache-dir -r requirements-api.txt
COPY app.py model.joblib ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```
Build & run:
```bash
docker build -t ds-60day-api .
docker run --rm -p 8000:8000 ds-60day-api
```
Test:
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H 'Content-Type: application/json' \
  -d '{"features": [5.1, 3.5, 1.4, 0.2]}'
```
PowerShell:
```powershell
$body = @{ features = @(5.1, 3.5, 1.4, 0.2) } | ConvertTo-Json
Invoke-RestMethod -Method Post -Uri 'http://127.0.0.1:8000/predict' -ContentType 'application/json' -Body $body
```


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — container build boundaries, minimal images, and health semantics

### Mental model

A container image is a versioned filesystem plus process configuration;
a container is one running process created from that image. Docker
builds layers from a **build context**, so `.dockerignore` controls what
can be sent to the builder. Multi-stage or ordered builds copy only the
runtime artifacts needed after dependency installation.

Liveness asks whether the process is functioning; readiness asks whether
it should receive traffic. Packaging does not add application security,
trusted artifacts, secret management, network policy, or safe defaults
automatically.

### Read the API before running it

- **`FROM` / pinned base:** establishes operating-system and Python dependencies; pin and scan rather than trusting `latest`.
- **`COPY` and `.dockerignore`:** define which local files enter the build context and image layers.
- **`CMD` / `HEALTHCHECK`:** define the main process and probe command without replacing service-level readiness policy.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — audit build-context inclusion before Docker runs

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** This simplified matcher demonstrates review logic; Docker's complete ignore semantics and the real context still need inspection.

In [ ]:
from fnmatch import fnmatch

files = [
    "app.py",
    "requirements.txt",
    ".env",
    ".venv/lib/package.py",
    "artifacts/model.joblib",
]
ignore_patterns = [".env", ".venv/*"]
included = [
    path for path in files
    if not any(fnmatch(path, pattern) for pattern in ignore_patterns)
]
print(included)
assert ".env" not in included and not any(p.startswith(".venv/") for p in included)

**Expected observation:** The environment file and local virtual environment are excluded while explicit runtime assets remain.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — distinguish process health from traffic readiness

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Restarting the healthy process would not repair the external dependency and could amplify the incident.

In [ ]:
def probe(*, process_running, model_loaded, dependency_ready, draining):
    return {
        "healthy": process_running,
        "ready": (
            process_running
            and model_loaded
            and dependency_ready
            and not draining
        ),
    }

during_dependency_outage = probe(
    process_running=True,
    model_loaded=True,
    dependency_ready=False,
    draining=False,
)
print(during_dependency_outage)
assert during_dependency_outage == {"healthy": True, "ready": False}

**Expected observation:** A running process can remain live while correctly refusing new traffic during a dependency outage.

### Debugging and practice ramp

**Common mistake:** Copying the whole repository into an image, baking credentials into layers, or binding a development server publicly.

**Diagnostic:** Inspect build context, image history/SBOM, user, exposed ports, process signals, environment, artifact hash, and health/readiness responses.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define container build boundaries, minimal images, and health semantics in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not publish an image until secrets, base/dependency provenance, non-root runtime, probes, resource limits, and rollback are reviewed.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Create a slim dependency file containing only direct API runtime needs.

**Verify:** Practice 1 — container build boundaries, minimal images, and health semantics — build the image from a direct-runtime-only dependency file, print resolved package versions and image size, and run the health/predict smoke tests; prove test/notebook-only packages are absent.

2. Add `GET /health` returning `{"status": "ok"}`.

**Verify:** Practice 2 — container build boundaries, minimal images, and health semantics — with TestClient and the running container, assert GET /health returns status 200 and exactly {'status': 'ok'}; distinguish liveness from readiness by testing a missing/tampered model artifact.

3. Optionally push the image to a registry if you intentionally use a connected
   account.

**Verify:** Practice 3 — container build boundaries, minimal images, and health semantics — keep this optional and connected: either record a skipped result, or name the registry/repository/tag/digest, show authenticated push exit code 0, pull by digest, and rerun health/predict without exposing credentials.

### Progressive hints

1. Trace imports from `app.py` and the serialized pipeline. Rebuild in a clean
   image and run both endpoints.
2. Keep liveness cheap. A Docker `HEALTHCHECK` can use Python's standard
   `urllib.request` so a slim image does not need `curl`.
3. Use a non-secret image name/tag, authenticate through the registry's
   supported credential flow, scan the image, and never embed credentials in a
   layer. Registry upload is optional and networked.

### Additional mastery practice

Containerize a minimal, testable service without embedding secrets or privileged assumptions. Build, readiness, and runtime health have distinct contracts.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Layer and secret audit:** Create a `.dockerignore`, inspect image history, and prove that `.env`, Git metadata, notebooks, caches, and local artifacts are absent.
   **Progressive hint:** The build context is the first boundary. Deleting a secret in a later layer does not remove it from earlier layers.

**Verify:** Layer and secret audit — build/save the image, inspect history and archive file list, and assert sentinel .env, .git, notebook, cache, and artifacts paths/content are absent while required application files remain.

5. **Least-privilege runtime:** Run the service as a non-root user with a read-only filesystem and an explicit writable temporary directory. Diagnose any write assumptions.
   **Progressive hint:** Create the user in the image, set ownership only where needed, and write transient files under an intentionally mounted/temp path.

**Verify:** Least-privilege runtime — inside the container, print UID/GID and filesystem mount policy; assert UID is nonzero, writes outside the declared temp path fail, temp writes succeed, and health/predict still return expected statuses.

6. **Health semantics:** Implement separate `/live` and `/ready` checks and a startup failure when the model manifest is incompatible. Test all three states.
   **Progressive hint:** Liveness answers whether the process can respond; readiness answers whether it can safely serve the declared model contract.

**Verify:** Health semantics — assert /live is 200 while the process runs, /ready is 200 only after a compatible artifact loads, and tampered/missing manifests produce non-ready or startup failure with a sanitized message.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Layer and secret audit


# Practice 5 — Least-privilege runtime


# Practice 6 — Health semantics
